# 🌙 Night Economy Taxi Agent — Grid London AI

**Module:** Artificial Intelligence (CMP-N206-0)  
**Programme:** BSc Computer Science  
**University:** University of Roehampton  
**Student Name:** Baburam Bastola 
**Student ID:** A00022220

---

## Project Overview

This notebook implements a **rational AI taxi agent** operating in a grid-based simulation of London during Friday and Saturday night hours. The agent navigates a 10×10 grid representing real London nightlife areas (Soho, Shoreditch, Camden, Brixton), picks up passengers, and delivers them to home zones while maximising a utility score.

The agent is designed around the **agent + environment + rationality** paradigm described in Russell & Norvig [1], and uses **BFS** and **A\*** search strategies to find optimal paths.

---

## Notebook Structure

| Section | Content |
|---|---|
| **1. Setup** | Imports and configuration |
| **2. Environment** | Grid world, cell types, London map |
| **3. Passenger** | Passenger class with drunk logic |
| **4. Agent** | TaxiAgent class with percepts and actions |
| **5. Performance Measure** | Scoring system and utility function |
| **6. Search — BFS** | Breadth-First Search pathfinding |
| **7. Search — A\*** | A* with Manhattan distance heuristic |
| **8. Explainability** | Agent reasoning output |
| **9. Scenario 1** | Normal Friday night |
| **10. Scenario 2** | Heavy road closures |
| **11. Scenario 3** | Drunk passenger surge night |
| **12. Evaluation** | Results comparison and visualisations |
| **13. Reflection** | Fairness, sustainability, limitations |
| **14. References** | IEEE formatted sources |

---
## Section 1 — Setup & Imports

In [ ]:
# ============================================================
# SECTION 1: IMPORTS AND CONFIGURATION
# All standard libraries — no pip install needed
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
import matplotlib.patheffects as pe
from collections import deque
import heapq
import random
import time
import copy
import os

# Reproducibility
random.seed(42)
np.random.seed(42)

os.makedirs('assets', exist_ok=True)

print('✅ All libraries loaded successfully')
print('🌙 Night Economy Taxi Agent — Ready to initialise')

---
## Section 2 — Environment: Grid-London World

The environment is a **10×10 grid** representing central London at night. Each cell has a type:

- `0` = Normal road (white) — drivable, no penalty
- `1` = Blocked cell (red) — road closure, crowd, accident — cannot enter, −5 if attempted
- `2` = Hotspot / Surge zone (yellow) — Soho, Shoreditch, Camden, Brixton — +5 bonus on pickup
- `3` = Home zone (green) — Islington, Hackney, Clapham, Richmond — common drop-off destinations

Following Russell & Norvig's environment taxonomy [1], this environment is: **partially observable**, **stochastic**, **sequential**, **static**, **discrete**, and **single-agent**.

In [ ]:
# ============================================================
# SECTION 2: GRID WORLD — 10x10 GRID-LONDON
# ============================================================

# Cell type constants
NORMAL  = 0   # Standard road
BLOCKED = 1   # Road closure / crowd
HOTSPOT = 2   # Surge zone (Soho, Shoreditch, Camden, Brixton)
HOME    = 3   # Home zone (Islington, Hackney, Clapham, Richmond)

AREA_LABELS = [
    (0, 1, 'Soho'),
    (0, 6, 'Shoreditch'),
    (2, 0, 'Camden'),
    (9, 2, 'Brixton'),
    (3, 9, 'Islington'),
    (6, 2, 'Hackney'),
    (7, 4, 'Clapham'),
    (9, 8, 'Richmond'),
]

class GridWorld:
    """
    Represents the Grid-London environment.
    A 10x10 grid where each cell has a type (NORMAL, BLOCKED, HOTSPOT, HOME).
    """

    def __init__(self, blocked_cells=None):
        self.rows = 10
        self.cols = 10
        self.grid = np.zeros((self.rows, self.cols), dtype=int)

        # Hotspot zones (surge pricing)
        self.hotspot_cells = [
            (0, 1), (0, 6), (2, 0), (9, 2), (4, 3), (6, 5),
        ]
        for r, c in self.hotspot_cells:
            self.grid[r][c] = HOTSPOT

        # Home zones (drop-off destinations)
        self.home_cells = [
            (3, 9), (6, 2), (7, 4), (9, 8), (1, 8), (5, 0),
        ]
        for r, c in self.home_cells:
            self.grid[r][c] = HOME

        # Blocked cells (road closures) — default scenario
        if blocked_cells is None:
            self.blocked_cells = [
                (0, 3), (1, 2), (2, 4),
                (3, 1), (4, 5), (5, 7),
                (7, 1), (8, 3),
            ]
        else:
            self.blocked_cells = blocked_cells

        for r, c in self.blocked_cells:
            self.grid[r][c] = BLOCKED

    def is_valid(self, row, col):
        return 0 <= row < self.rows and 0 <= col < self.cols

    def is_blocked(self, row, col):
        return self.grid[row][col] == BLOCKED

    def is_hotspot(self, row, col):
        return self.grid[row][col] == HOTSPOT

    def is_home(self, row, col):
        return self.grid[row][col] == HOME

    def get_neighbours(self, row, col):
        """Return valid non-blocked neighbours with direction labels."""
        directions = [(-1,0,'N'),(1,0,'S'),(0,1,'E'),(0,-1,'W')]
        neighbours = []
        for dr, dc, direction in directions:
            nr, nc = row + dr, col + dc
            if self.is_valid(nr, nc) and not self.is_blocked(nr, nc):
                neighbours.append((nr, nc, direction))
        return neighbours

    def render(self, agent_pos=None, passenger_pos=None, dest_pos=None,
               path=None, title='Grid-London Night Economy', ax=None):
        """Render the grid using matplotlib with colour-coded cells."""
        cmap = ListedColormap(['#1a2a3a', '#c0392b', '#f39c12', '#27ae60'])
        show = ax is None
        if show:
            fig, ax = plt.subplots(figsize=(10, 10))
            fig.patch.set_facecolor('#0c1929')

        ax.set_facecolor('#0c1929')
        ax.imshow(self.grid, cmap=cmap, vmin=0, vmax=3, alpha=0.85)

        for x in range(self.cols + 1):
            ax.axvline(x - 0.5, color='#1e3a5f', linewidth=0.5, alpha=0.5)
        for y in range(self.rows + 1):
            ax.axhline(y - 0.5, color='#1e3a5f', linewidth=0.5, alpha=0.5)

        if path and len(path) > 1:
            path_rows = [p[0] for p in path]
            path_cols = [p[1] for p in path]
            ax.plot(path_cols, path_rows, color='#38bdf8',
                    linewidth=2.5, alpha=0.7, zorder=3,
                    linestyle='--', marker='o', markersize=4)

        for r, c, label in AREA_LABELS:
            ax.text(c, r, label, ha='center', va='center',
                    fontsize=6.5, color='white', alpha=0.6, fontweight='bold')

        for r in range(self.rows):
            for c in range(self.cols):
                if self.grid[r][c] == BLOCKED:
                    ax.text(c, r, 'X', ha='center', va='center',
                            fontsize=12, color='#e74c3c', alpha=0.8)

        if passenger_pos and passenger_pos != agent_pos:
            ax.plot(passenger_pos[1], passenger_pos[0], 'o',
                    color='#f472b6', markersize=18, zorder=5, alpha=0.9)
            ax.text(passenger_pos[1], passenger_pos[0], 'P',
                    ha='center', va='center', fontsize=10,
                    color='white', fontweight='bold', zorder=6)

        if dest_pos:
            ax.plot(dest_pos[1], dest_pos[0], 's',
                    color='#34d399', markersize=18, zorder=5, alpha=0.9)
            ax.text(dest_pos[1], dest_pos[0], 'D',
                    ha='center', va='center', fontsize=10,
                    color='white', fontweight='bold', zorder=6)

        if agent_pos:
            ax.plot(agent_pos[1], agent_pos[0], 'D',
                    color='#38bdf8', markersize=20, zorder=7, alpha=0.95)
            ax.text(agent_pos[1], agent_pos[0], 'T',
                    ha='center', va='center', fontsize=11,
                    color='white', fontweight='bold', zorder=8)

        ax.set_xticks(range(self.cols))
        ax.set_yticks(range(self.rows))
        ax.set_xticklabels(range(self.cols), color='#7dd3fc', fontsize=9)
        ax.set_yticklabels(range(self.rows), color='#7dd3fc', fontsize=9)
        ax.tick_params(colors='#7dd3fc')
        for spine in ax.spines.values():
            spine.set_edgecolor('#1e3a5f')

        legend_elements = [
            mpatches.Patch(color='#1a2a3a', label='Normal road'),
            mpatches.Patch(color='#c0392b', label='Blocked (road closure)'),
            mpatches.Patch(color='#f39c12', label='Hotspot (surge zone)'),
            mpatches.Patch(color='#27ae60', label='Home zone'),
        ]
        ax.legend(handles=legend_elements, loc='upper right',
                  facecolor='#0c1929', edgecolor='#1e3a5f',
                  labelcolor='white', fontsize=8)
        ax.set_title(title, color='#7dd3fc', fontsize=13,
                     fontweight='bold', pad=12)

        if show:
            plt.tight_layout()
            safe_title = title.replace(' ', '_').replace('/', '-')
            plt.savefig(f'assets/{safe_title}.png',
                        dpi=120, bbox_inches='tight', facecolor='#0c1929')
            plt.show()


# Initialise and render
world = GridWorld()
print('✅ GridWorld created')
print(f'   Grid size: {world.rows} x {world.cols}')
print(f'   Hotspot cells: {world.hotspot_cells}')
print(f'   Home cells:    {world.home_cells}')
print(f'   Blocked cells: {world.blocked_cells}')
print()
world.render(title='Grid-London Night Economy Base Map')

---
## Section 3 — Passenger Class

The `Passenger` class models a real London night-time taxi passenger. On Friday and Saturday nights, approximately 30% of late-night passengers are intoxicated [2], causing delays for drivers. This is modelled as a 30% chance of `is_drunk = True`, which adds a 2-step delay and a -2 score penalty.

In [ ]:
# ============================================================
# SECTION 3: PASSENGER CLASS
# ============================================================

class Passenger:
    """
    Represents a night-time taxi passenger in Grid-London.

    Attributes:
        id          : unique passenger identifier
        location    : (row, col) where passenger is waiting
        destination : (row, col) where they want to go
        is_drunk    : bool — 30% chance True on Friday/Saturday night
        wait_time   : steps the passenger has been waiting
    """

    _id_counter = 0

    def __init__(self, location, destination, is_drunk=None):
        Passenger._id_counter += 1
        self.id          = Passenger._id_counter
        self.location    = location
        self.destination = destination
        self.wait_time   = 0

        if is_drunk is None:
            self.is_drunk = random.random() < 0.30
        else:
            self.is_drunk = is_drunk

    def __str__(self):
        drunk_str = 'DRUNK (+2 step delay)' if self.is_drunk else 'Sober'
        return (f'Passenger #{self.id} | Location: {self.location} | '
                f'Destination: {self.destination} | Status: {drunk_str}')


def spawn_passenger(world, force_drunk=False, force_sober=False):
    """
    Spawn a passenger at a random hotspot with a random home destination.
    Mirrors real taxi dispatch — passengers appear at nightlife venues
    and want to go home.
    """
    location    = random.choice(world.hotspot_cells)
    destination = random.choice(world.home_cells)

    while destination == location:
        destination = random.choice(world.home_cells)

    if force_drunk:
        is_drunk = True
    elif force_sober:
        is_drunk = False
    else:
        is_drunk = None

    return Passenger(location, destination, is_drunk=is_drunk)


# Test
Passenger._id_counter = 0
test_p = spawn_passenger(world)
print('✅ Passenger class working')
print(test_p)

---
## Section 4 — Agent Class: TaxiAgent

The `TaxiAgent` is a **goal-based rational agent** [1]. It maintains an internal model of the world, perceives its state, and chooses actions to maximise its utility score.

In [ ]:
# ============================================================
# SECTION 4: TAXI AGENT CLASS
# ============================================================

class TaxiAgent:
    """
    A goal-based rational taxi agent operating in Grid-London.

    Perceives environment, plans routes using BFS or A*,
    executes actions to pick up and deliver passengers,
    and maximises a utility score.

    Agent type: Goal-based (Russell & Norvig, 2020, p.56)
    """

    def __init__(self, world, start_pos=(0, 0)):
        self.world          = world
        self.position       = start_pos
        self.score          = 0
        self.has_passenger  = False
        self.passenger      = None
        self.trips_complete = 0
        self.total_steps    = 0
        self.score_log      = []
        self.path_taken     = [start_pos]
        self.explain_log    = []

    # --- PERCEPTS ---

    def perceive(self):
        """Return agent current perceptual state as a dictionary."""
        return {
            'position'          : self.position,
            'has_passenger'     : self.has_passenger,
            'passenger_location': self.passenger.location if self.passenger else None,
            'destination'       : self.passenger.destination if self.passenger else None,
            'passenger_drunk'   : self.passenger.is_drunk if self.passenger else None,
            'blocked_cells'     : self.world.blocked_cells,
            'current_score'     : self.score,
            'trips_complete'    : self.trips_complete,
        }

    # --- SCORING ---

    def _log_score(self, change, reason):
        self.score += change
        self.score_log.append({
            'step': self.total_steps, 'change': change,
            'reason': reason, 'total': self.score
        })

    # --- ACTIONS ---

    def _move(self, dr, dc, direction_name):
        """Internal move. Checks boundaries and blocked cells. Costs -1."""
        nr = self.position[0] + dr
        nc = self.position[1] + dc

        if not self.world.is_valid(nr, nc):
            print(f'  [BLOCKED] Cannot move {direction_name} — outside boundary')
            return False

        if self.world.is_blocked(nr, nc):
            self._log_score(-5, f'Hit blocked cell attempting {direction_name}')
            print(f'  [BLOCKED] {direction_name} to ({nr},{nc}) blocked. Penalty -5')
            return False

        self.position = (nr, nc)
        self.total_steps += 1
        self._log_score(-1, f'Move {direction_name} to {self.position}')
        self.path_taken.append(self.position)
        return True

    def move_north(self): return self._move(-1,  0, 'North')
    def move_south(self): return self._move( 1,  0, 'South')
    def move_east(self):  return self._move( 0,  1, 'East')
    def move_west(self):  return self._move( 0, -1, 'West')

    def pickup(self):
        """Pick up passenger. Agent must be on passenger cell."""
        if not self.passenger:
            print('[PICKUP FAILED] No passenger assigned.')
            return False

        if self.position != self.passenger.location:
            print(f'[PICKUP FAILED] Agent at {self.position}, '
                  f'passenger at {self.passenger.location}')
            return False

        self.has_passenger = True

        if self.world.is_hotspot(*self.position):
            self._log_score(+5, 'Surge zone pickup bonus')
            print(f'  Surge zone pickup! +5 bonus')

        if self.passenger.is_drunk:
            self._log_score(-2, 'Drunk passenger delay')
            self.total_steps += 2
            print(f'  Drunk passenger — 2-step delay, -2 score')

        print(f'  Picked up Passenger #{self.passenger.id} at {self.position}')
        return True

    def dropoff(self):
        """Drop off passenger at destination. Awards +20. Checks 3-trip bonus."""
        if not self.has_passenger:
            print('[DROPOFF FAILED] No passenger on board.')
            return False

        if self.position != self.passenger.destination:
            print(f'[DROPOFF FAILED] Agent at {self.position}, '
                  f'destination is {self.passenger.destination}')
            return False

        self._log_score(+20, f'Successful dropoff Passenger #{self.passenger.id}')
        self.has_passenger = False
        self.trips_complete += 1

        print(f'  Dropped off Passenger #{self.passenger.id} at {self.position}. +20')

        if self.trips_complete == 3:
            self._log_score(+10, 'Three-trip session completion bonus')
            print(f'  Three trips complete! Bonus: +10')

        self.passenger = None
        return True

    def print_score_log(self):
        print('=' * 55)
        print('SCORE LOG')
        print('=' * 55)
        for event in self.score_log:
            sign = '+' if event['change'] >= 0 else ''
            print(f"  Step {event['step']:3d} | "
                  f"{sign}{event['change']:3d} | "
                  f"Total: {event['total']:4d} | "
                  f"{event['reason']}")
        print('=' * 55)
        print(f'  FINAL SCORE: {self.score}')
        print(f'  TOTAL STEPS: {self.total_steps}')
        print(f'  TRIPS DONE:  {self.trips_complete}')
        print('=' * 55)

    def __str__(self):
        status = (f'Passenger #{self.passenger.id} on board'
                  if self.has_passenger else 'No passenger')
        return (f'TaxiAgent | Position: {self.position} | '
                f'Score: {self.score} | Steps: {self.total_steps} | '
                f'Trips: {self.trips_complete} | {status}')


# Test
agent = TaxiAgent(world, start_pos=(5, 5))
print('✅ TaxiAgent created')
print(agent)
print()
print('Percept state:')
for k, v in agent.perceive().items():
    print(f'  {k}: {v}')

---
## Section 5 — Performance Measure

The performance measure is the **single most important design decision** in a rational agent [1].

| Event | Score | Justification |
|---|---|---|
| Successful drop-off | +20 | Primary revenue — mirrors real taxi fare |
| Each move (fuel cost) | -1 | Incentivises shortest path — also a CO2 signal |
| Attempting blocked cell | -5 | Wasted time and potential vehicle damage |
| Drunk passenger delay | -2 | Real driver time loss with difficult passengers |
| Surge zone pickup | +5 | Mirrors Uber/Bolt surge pricing |
| 3-trip completion bonus | +10 | Rewards sustained session efficiency |

In [ ]:
# ========================================
# SECTION 5: PERFORMANCE MEASURE VISUALISATION
# Scoring is built into the agent — this cell visualises it
# ============================================================

fig, ax = plt.subplots(figsize=(10, 4))
fig.patch.set_facecolor('#0c1929')
ax.set_facecolor('#0c1929')

events = ['Drop-off (+20)', 'Surge Bonus (+5)', '3-Trip Bonus (+10)',
          'Blocked Attempt (-5)', 'Move Cost (-1)', 'Drunk Delay (-2)']
values = [20, 5, 10, -5, -1, -2]
colours = ['#34d399' if v > 0 else '#f87171' for v in values]

bars = ax.bar(events, values, color=colours, edgecolor='#1e3a5f',
              linewidth=1.2, alpha=0.85, width=0.6)

for bar, val in zip(bars, values):
    ypos = bar.get_height() + (0.5 if val >= 0 else -1.5)
    ax.text(bar.get_x() + bar.get_width()/2, ypos,
            f'{val:+d}', ha='center', va='bottom',
            color='white', fontweight='bold', fontsize=11)

ax.axhline(0, color='#7dd3fc', linewidth=0.8, alpha=0.4)
ax.set_title('Performance Measure — Utility Function', color='#7dd3fc',
             fontsize=13, fontweight='bold', pad=10)
ax.set_ylabel('Score Change', color='#7dd3fc', fontsize=10)
ax.tick_params(colors='#7dd3fc', labelsize=8)
for spine in ax.spines.values():
    spine.set_edgecolor('#1e3a5f')

plt.tight_layout()
plt.savefig('assets/performance_measure.png', dpi=120,
            bbox_inches='tight', facecolor='#0c1929')
plt.show()
print('✅ Performance measure visualised')

---
## Section 6 — Search Strategy: Breadth-First Search (BFS)

BFS explores all nodes at depth d before depth d+1, guaranteeing the **shortest path** in an unweighted grid [1].

- **Time complexity:** O(b^d) where b=4 (branching factor), d=depth
- **Completeness:** Yes — always finds a path if one exists
- **Optimality:** Yes — shortest path in unweighted grid
- **Why BFS here:** Every move costs -1 equally, so shortest path = cheapest path

In [ ]:
# ============================================================
# SECTION 6: BFS PATHFINDING
# ============================================================

def bfs(world, start, goal):
    """
    Breadth-First Search pathfinding on the GridWorld.

    Finds the shortest path from start to goal, avoiding blocked cells.
    Uses a deque (double-ended queue) for efficient O(1) popleft.

    Args:
        world : GridWorld
        start : (row, col) starting position
        goal  : (row, col) target position

    Returns:
        (path, moves, nodes_explored)
        path           : list of (row,col) from start to goal
        moves          : list of direction strings ('N','S','E','W')
        nodes_explored : int
        Returns (None, None, nodes) if no path exists.
    """
    if start == goal:
        return [start], [], 0

    queue   = deque([(start, [start], [])])
    visited = {start}
    nodes   = 0

    while queue:
        current, path, moves = queue.popleft()
        nodes += 1

        for nr, nc, direction in world.get_neighbours(*current):
            next_pos = (nr, nc)
            if next_pos not in visited:
                new_path  = path + [next_pos]
                new_moves = moves + [direction]

                if next_pos == goal:
                    return new_path, new_moves, nodes

                visited.add(next_pos)
                queue.append((next_pos, new_path, new_moves))

    return None, None, nodes


# Test BFS
path_bfs, moves_bfs, nodes_bfs = bfs(world, (0,0), (9,9))
print('✅ BFS working')
print(f'  Start: (0,0)  ->  Goal: (9,9)')
if path_bfs:
    print(f'  Path length   : {len(path_bfs) - 1} moves')
    print(f'  Moves         : {" ".join(moves_bfs)}')
    print(f'  Nodes explored: {nodes_bfs}')
    print(f'  Est. score    : -{len(moves_bfs)} (fuel) + 20 (dropoff) = {20 - len(moves_bfs)}')
else:
    print('  No path found!')

---
## Section 7 — Search Strategy: A* (Distinction Level)

A* uses `f(n) = g(n) + h(n)` where `g(n)` is the cost so far and `h(n)` is the **Manhattan distance heuristic** [1, 6].

- **Heuristic:** `h(n) = |row_current - row_goal| + |col_current - col_goal|`
- **Admissible?** Yes — Manhattan distance never overestimates in a grid
- **Advantage:** Explores far fewer nodes than BFS by prioritising promising directions
- **When it fails:** Dense blocked layouts where the heuristic is misleading

In [ ]:
# ============================================================
# SECTION 7: A* PATHFINDING
# ============================================================

def manhattan(a, b):
    """Manhattan distance heuristic — admissible for grid worlds."""
    return abs(a[0] - b[0]) + abs(a[1] - b[1])


def astar(world, start, goal):
    """
    A* Search on the GridWorld with Manhattan distance heuristic.
    Uses heapq (min-heap priority queue).

    Returns: (path, moves, nodes_explored) — same format as bfs()
    """
    if start == goal:
        return [start], [], 0

    h_start = manhattan(start, goal)
    heap    = [(h_start, 0, start, [start], [])]
    visited = {}
    nodes   = 0

    while heap:
        f, g, current, path, moves = heapq.heappop(heap)
        nodes += 1

        if current == goal:
            return path, moves, nodes

        if current in visited and visited[current] <= g:
            continue
        visited[current] = g

        for nr, nc, direction in world.get_neighbours(*current):
            next_pos = (nr, nc)
            new_g    = g + 1
            new_h    = manhattan(next_pos, goal)
            new_f    = new_g + new_h

            if next_pos not in visited or visited[next_pos] > new_g:
                heapq.heappush(heap, (
                    new_f, new_g, next_pos,
                    path + [next_pos],
                    moves + [direction]
                ))

    return None, None, nodes


# Compare BFS vs A*
test_pairs = [((0,0),(9,9)), ((0,0),(3,9)), ((2,0),(9,8))]

print('✅ A* working')
print()
print(f'{"Start->Goal":<18} {"BFS Nodes":>10} {"A* Nodes":>10} {"BFS Len":>8} {"A* Len":>8}')
print('-' * 58)

for start_t, goal_t in test_pairs:
    _, m_bfs, n_bfs = bfs(world, start_t, goal_t)
    _, m_ast, n_ast = astar(world, start_t, goal_t)
    bfs_len = len(m_bfs) if m_bfs else 'N/A'
    ast_len = len(m_ast) if m_ast else 'N/A'
    label   = f'{start_t}->{goal_t}'
    print(f'{label:<18} {n_bfs:>10} {n_ast:>10} {str(bfs_len):>8} {str(ast_len):>8}')

print()
print('A* explores fewer nodes because the Manhattan heuristic')
print('guides search toward the goal rather than exploring evenly.')

---
## Section 8 — Explainability & Trip Runner

After every move the agent explains its decision. This is a fundamental requirement for **trustworthy AI** — the agent must be auditable [3]. The `run_trip()` function ties everything together: plans the path, executes each move, and prints structured reasoning at every step.

In [ ]:
# ============================================================
# SECTION 8: EXPLAINABILITY + FULL TRIP RUNNER
# ============================================================

DIRECTION_FULL = {'N': 'North', 'S': 'South', 'E': 'East', 'W': 'West'}
ACTION_MAP     = {'N': 'move_north', 'S': 'move_south',
                  'E': 'move_east',  'W': 'move_west'}


def run_trip(agent, passenger, search='bfs', verbose=True):
    """
    Run one complete trip:
      1. Navigate to passenger (BFS or A*)
      2. Pick up passenger (handle drunk delay, surge bonus)
      3. Navigate to destination
      4. Drop off passenger (check 3-trip bonus)

    After every move: print direction, reason, remaining path, current score.

    Args:
        agent     : TaxiAgent
        passenger : Passenger
        search    : 'bfs' or 'astar'
        verbose   : print step-by-step explainability output

    Returns:
        dict with trip results
    """
    agent.passenger  = passenger
    search_fn        = bfs if search == 'bfs' else astar
    search_name      = 'BFS' if search == 'bfs' else 'A*'
    trip_start_score = agent.score
    trip_start_steps = agent.total_steps

    if verbose:
        print(f'{"="*60}')
        print(f'TRIP #{agent.trips_complete + 1}  |  Search: {search_name}')
        print(f'  Agent start : {agent.position}')
        print(f'  Passenger   : {passenger.location} '
              f'({"DRUNK" if passenger.is_drunk else "Sober"})')
        print(f'  Destination : {passenger.destination}')
        print(f'{"="*60}')

    # PHASE 1: Navigate to passenger
    path_to_p, moves_to_p, nodes_1 = search_fn(
        agent.world, agent.position, passenger.location)

    if path_to_p is None:
        print('NO PATH to passenger — environment fully blocked!')
        return {'success': False, 'reason': 'No path to passenger',
                'total_score': agent.score, 'trip_steps': 0,
                'trip_score': 0, 'nodes_explored': 0}

    if verbose:
        print(f'\nPhase 1 — Navigate to passenger')
        print(f'  {search_name} path: {len(moves_to_p)} moves | '
              f'{nodes_1} nodes explored')

    for i, direction in enumerate(moves_to_p):
        prev_pos  = agent.position
        remaining = len(moves_to_p) - i - 1
        getattr(agent, ACTION_MAP[direction])()
        if verbose:
            print(f'  [STEP {agent.total_steps:3d}] '
                  f'{prev_pos} -> {agent.position} '
                  f'({DIRECTION_FULL[direction]}) | '
                  f'To pickup: {remaining} moves | Score: {agent.score}')

    # Pick up
    if verbose:
        print(f'\n  >> PICKUP')
    success = agent.pickup()
    if not success:
        return {'success': False, 'reason': 'Pickup failed',
                'total_score': agent.score, 'trip_steps': 0,
                'trip_score': 0, 'nodes_explored': nodes_1}

    # PHASE 2: Navigate to destination
    path_to_d, moves_to_d, nodes_2 = search_fn(
        agent.world, agent.position, passenger.destination)

    if path_to_d is None:
        print('NO PATH to destination!')
        return {'success': False, 'reason': 'No path to destination',
                'total_score': agent.score, 'trip_steps': agent.total_steps - trip_start_steps,
                'trip_score': agent.score - trip_start_score, 'nodes_explored': nodes_1}

    if verbose:
        print(f'\nPhase 2 — Navigate to destination')
        print(f'  {search_name} path: {len(moves_to_d)} moves | '
              f'{nodes_2} nodes explored')

    for i, direction in enumerate(moves_to_d):
        prev_pos  = agent.position
        remaining = len(moves_to_d) - i - 1
        getattr(agent, ACTION_MAP[direction])()
        if verbose:
            print(f'  [STEP {agent.total_steps:3d}] '
                  f'{prev_pos} -> {agent.position} '
                  f'({DIRECTION_FULL[direction]}) | '
                  f'To dest: {remaining} moves | Score: {agent.score}')

    # Drop off
    if verbose:
        print(f'\n  >> DROPOFF')
    agent.dropoff()

    trip_score = agent.score - trip_start_score
    trip_steps = agent.total_steps - trip_start_steps

    if verbose:
        print(f'\nTrip complete!')
        print(f'  Trip score  : {trip_score:+d}')
        print(f'  Trip steps  : {trip_steps}')
        print(f'  Total score : {agent.score}')

    return {
        'success'      : True,
        'trip_score'   : trip_score,
        'trip_steps'   : trip_steps,
        'total_score'  : agent.score,
        'nodes_explored': nodes_1 + nodes_2,
        'search'       : search_name,
    }


print('✅ run_trip() ready — explainability output built in to every step')

---
## Section 9 — Scenario 1: Normal Friday Night

**Setup:** Standard road closures (8 blocked cells), 1 sober passenger, BFS search.  
**Purpose:** Establish baseline performance for comparison.

In [ ]:
# ============================================================
# SECTION 9: SCENARIO 1 — NORMAL FRIDAY NIGHT
# ============================================================

print('SCENARIO 1 — Normal Friday Night')
print('8 blocked cells | 1 sober passenger | BFS')
print()

Passenger._id_counter = 0
world_s1     = GridWorld()
agent_s1     = TaxiAgent(world_s1, start_pos=(5, 5))
passenger_s1 = spawn_passenger(world_s1, force_sober=True)

print(passenger_s1)
print()

result_s1 = run_trip(agent_s1, passenger_s1, search='bfs', verbose=True)

print()
agent_s1.print_score_log()

world_s1.render(
    agent_pos=agent_s1.position,
    path=agent_s1.path_taken,
    title='Scenario 1 Normal Friday Night BFS Path'
)

print(f'\nScenario 1 Result:')
print(f'  Score:         {result_s1["total_score"]}')
print(f'  Steps taken:   {result_s1["trip_steps"]}')
print(f'  Goal achieved: {"YES" if result_s1["success"] else "NO"}')
print(f'  Nodes explored:{result_s1["nodes_explored"]}')

---
## Section 10 — Scenario 2: Heavy Road Closures

**Setup:** 15 blocked cells (nearly double normal), 1 sober passenger, BFS.  
**Purpose:** Test agent under extreme constraints. Score should be lower than Scenario 1.

In [ ]:
# ============================================================
# SECTION 10: SCENARIO 2 — HEAVY ROAD CLOSURES
# ============================================================

print('SCENARIO 2 — Heavy Road Closures')
print('15 blocked cells | 1 sober passenger | BFS')
print()

heavy_blocked = [
    (0, 3), (1, 2), (2, 4), (3, 1), (4, 5),
    (5, 7), (7, 1), (8, 3), (1, 5), (2, 7),
    (3, 4), (6, 7), (4, 8), (5, 3), (7, 6),
]

Passenger._id_counter = 0
world_s2     = GridWorld(blocked_cells=heavy_blocked)
agent_s2     = TaxiAgent(world_s2, start_pos=(5, 5))
passenger_s2 = spawn_passenger(world_s2, force_sober=True)

# Ensure passenger is not on a blocked cell
attempts = 0
while passenger_s2.location in heavy_blocked and attempts < 20:
    passenger_s2 = spawn_passenger(world_s2, force_sober=True)
    attempts += 1

print(passenger_s2)
print()

result_s2 = run_trip(agent_s2, passenger_s2, search='bfs', verbose=True)

print()
agent_s2.print_score_log()

world_s2.render(
    agent_pos=agent_s2.position,
    path=agent_s2.path_taken,
    title='Scenario 2 Heavy Road Closures BFS Path'
)

print(f'\nScenario 2 Result:')
if result_s2['success']:
    print(f'  Score:         {result_s2["total_score"]}')
    print(f'  Steps taken:   {result_s2["trip_steps"]}')
    print(f'  Goal achieved: YES')
    print(f'  Nodes explored:{result_s2["nodes_explored"]}')
    print(f'  Note: Score lower than S1 — longer detour due to more blocked cells')
else:
    print(f'  Goal achieved: NO — {result_s2["reason"]}')
    print(f'  This is valid — agent correctly reports no viable path')

---
## Section 11 — Scenario 3: Drunk Passenger Surge Night

**Setup:** 3 passengers (2 drunk), surge zones active, A* search.  
**Purpose:** Test multi-trip performance, drunk delays, surge bonuses. Should trigger 3-trip bonus.

In [ ]:
# ============================================================
# SECTION 11: SCENARIO 3 — DRUNK PASSENGER SURGE NIGHT
# ============================================================

print('SCENARIO 3 — Drunk Passenger Surge Night')
print('Standard closures | 3 passengers (2 drunk) | A*')
print()

Passenger._id_counter = 0
world_s3  = GridWorld()
agent_s3  = TaxiAgent(world_s3, start_pos=(0, 0))

# 3 passengers: sober, drunk, drunk
passengers_s3 = [
    spawn_passenger(world_s3, force_sober=True),
    spawn_passenger(world_s3, force_drunk=True),
    spawn_passenger(world_s3, force_drunk=True),
]

print('Passengers this session:')
for p in passengers_s3:
    print(f'  {p}')
print()

results_s3 = []
for i, p in enumerate(passengers_s3):
    print(f'\n{"#"*60}')
    print(f'# PASSENGER {i+1} OF {len(passengers_s3)}')
    print(f'{"#"*60}')
    r = run_trip(agent_s3, p, search='astar', verbose=True)
    results_s3.append(r)

print()
agent_s3.print_score_log()

world_s3.render(
    agent_pos=agent_s3.position,
    path=agent_s3.path_taken,
    title='Scenario 3 Surge Night A Star Path'
)

print(f'\nScenario 3 Results:')
for i, r in enumerate(results_s3):
    status = 'YES' if r['success'] else 'NO'
    print(f'  Trip {i+1}: Score {r.get("trip_score","N/A"):+} | '
          f'Steps {r.get("trip_steps","N/A")} | Goal: {status}')
print(f'  FINAL TOTAL SCORE: {agent_s3.score}')

---
## Section 12 — Evaluation: Results Comparison & Visualisations

In [ ]:
# ============================================================
# SECTION 12: EVALUATION — COMPARISON CHARTS
# ============================================================

# Collect results from all scenarios
scenarios_data = [
    {
        'name': 'S1: Normal Night (BFS)',
        'score': result_s1['total_score'],
        'steps': result_s1['trip_steps'],
        'nodes': result_s1['nodes_explored'],
        'search': 'BFS',
        'success': result_s1['success'],
    },
    {
        'name': 'S2: Heavy Closures (BFS)',
        'score': result_s2.get('total_score', 0),
        'steps': result_s2.get('trip_steps', 0),
        'nodes': result_s2.get('nodes_explored', 0),
        'search': 'BFS',
        'success': result_s2['success'],
    },
    {
        'name': 'S3: Surge Night (A*)',
        'score': agent_s3.score,
        'steps': agent_s3.total_steps,
        'nodes': sum(r.get('nodes_explored', 0) for r in results_s3),
        'search': 'A*',
        'success': all(r['success'] for r in results_s3),
    },
]

labels  = [s['name']  for s in scenarios_data]
scores  = [s['score'] for s in scenarios_data]
steps   = [s['steps'] for s in scenarios_data]
nodes   = [s['nodes'] for s in scenarios_data]
colours = ['#34d399', '#f87171', '#f472b6']

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('#0c1929')

def style_ax(ax, title, ylabel):
    ax.set_facecolor('#0c1929')
    ax.set_title(title, color='#7dd3fc', fontsize=10, fontweight='bold', pad=8)
    ax.set_ylabel(ylabel, color='#7dd3fc', fontsize=9)
    ax.tick_params(colors='#7dd3fc', labelsize=7)
    for spine in ax.spines.values():
        spine.set_edgecolor('#1e3a5f')

for ax, data, title, ylabel in zip(
    axes,
    [scores, steps, nodes],
    ['Total Score', 'Steps Taken', 'Nodes Explored'],
    ['Score', 'Steps', 'Nodes']
):
    bars = ax.bar(labels, data, color=colours, edgecolor='#1e3a5f',
                  alpha=0.85, width=0.5)
    for bar, val in zip(bars, data):
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + max(data)*0.01,
                str(val), ha='center', color='white',
                fontsize=9, fontweight='bold')
    ax.axhline(0, color='#7dd3fc', linewidth=0.5, alpha=0.3)
    style_ax(ax, title, ylabel)

plt.suptitle('Night Economy Taxi Agent — Evaluation Results',
             color='#e0f2fe', fontsize=12, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('assets/evaluation_results.png', dpi=120,
            bbox_inches='tight', facecolor='#0c1929')
plt.show()

# Text summary table
print('\nEVALUATION SUMMARY')
print(f'{"Scenario":<30} {"Score":>7} {"Steps":>7} {"Goal":>6} {"Search":>7}')
print('-' * 62)
for s in scenarios_data:
    goal = "YES" if s["success"] else "NO"
    print(f'{s["name"]:<30} {s["score"]:>7} {s["steps"]:>7} {goal:>6} {s["search"]:>7}')

---
## Section 13 — Reflection: Fairness, Inclusivity & Sustainability

### Fairness
The current performance measure could embed **geographic bias**. The agent earns +5 for pickups in Soho and Shoreditch (wealthier, central areas) but no bonus elsewhere. A fairer measure would include a **wait-time penalty** that grows regardless of passenger location, forcing equitable service [4].

### Inclusivity
The drunk passenger penalty (-2) deserves ethical scrutiny. Penalising the agent for picking up intoxicated passengers could lead a more advanced system to avoid vulnerable people — a pattern documented in rideshare discrimination research [4]. A responsible redesign treats drunk passengers with neutral score impact.

### Sustainability
Every move costs -1 — this is both an economic and **carbon signal**. Shortest-path optimisation directly reduces fuel and CO2 emissions. This aligns with TfL's zero-emission taxi target for 2033 [5]. A future extension could penalise routes through Clean Air Zones.

In [ ]:
# ============================================================
# SECTION 13: LIMITATIONS SUMMARY
# ============================================================

limitations = [
    ('Grid simplification',
     'Real London streets are not a perfect grid. A graph-based map '
     '(e.g. OSMnx + OpenStreetMap) would model one-way streets, '
     'roundabouts, and road hierarchies far more accurately.'),
    ('Static environment',
     'Blocked cells do not change during a trip. Real road closures '
     'appear and clear dynamically — a real agent would re-plan mid-journey.'),
    ('Single agent',
     'Only one taxi operates. Real dispatch involves fleet optimisation '
     'across many vehicles — a multi-agent problem with coordination challenges.'),
    ('No learning',
     'The agent does not improve from experience. A reinforcement learning '
     'extension could allow it to learn optimal pickup strategies over time.'),
    ('Binary drunk modelling',
     'Intoxication is binary (drunk/sober). Real-world passenger behaviour '
     'is a spectrum and ethically complex to model in a performance measure.'),
]

print('AGENT LIMITATIONS')
print('=' * 60)
for title, desc in limitations:
    print(f'\n  [{title}]')
    words = desc.split()
    line  = '  '
    for word in words:
        if len(line) + len(word) > 65:
            print(line)
            line = '  ' + word + ' '
        else:
            line += word + ' '
    print(line)
print('\n' + '=' * 60)

---
## Section 14 — References (IEEE Format)

[1] S. Russell and P. Norvig, *Artificial Intelligence: A Modern Approach*, 4th ed. Hoboken, NJ: Pearson, 2020.

[2] Greater London Authority, *London's Night-Time Economy*, GLA Economics, London, 2017. [Online]. Available: https://www.london.gov.uk

[3] European Commission, *Ethics Guidelines for Trustworthy AI*, High-Level Expert Group on Artificial Intelligence, Brussels, 2019. [Online]. Available: https://digital-strategy.ec.europa.eu

[4] K. Lum and W. Isaac, To predict and serve? *Significance*, vol. 13, no. 5, pp. 14-19, Oct. 2016.

[5] Transport for London, *Zero Emission Capable Taxis*, TfL, London, 2023. [Online]. Available: https://tfl.gov.uk

[6] P. Hart, N. Nilsson, and B. Raphael, A formal basis for the heuristic determination of minimum cost paths, *IEEE Transactions on Systems Science and Cybernetics*, vol. 4, no. 2, pp. 100-107, Jul. 1968.

---

## AI Use Declaration

This notebook was developed with the assistance of Claude (Anthropic) for conceptual scaffolding, code structure suggestions, and debugging support. All code has been reviewed, understood, and verified by the author. Every design decision can be explained during the in-person demo. AI use is declared in accordance with the University of Roehampton Student Guidelines on the Use of AI.